# TS-SatFire 논문의 Progression Prediction 예시 사건

**논문 예시:** arXiv Figure 8 / Scientific Data Figure 11,
Fire ID `NM3323810847220210520`.
**데이터 폴더 ID:** `US_2021_NM3323810847220210520` (FP 시험 분할).

[arXiv Figure 8 및 설명](https://arxiv.org/html/2412.11555v1#S4.F8) ·
[원본 예시 그림](https://arxiv.org/html/2412.11555v1/figures/pred2.jpeg) ·
[학술지판](https://www.nature.com/articles/s41597-025-06271-3)

이 노트북은 같은 사건의 실제 GeoTIFF를 이틀 입력(t−1,t)과 다음 날(t+1)으로
살펴봅니다. 원 논문의 SwinUNETR 예측·오분류 색상이나 256×256 crop을 재현하지 않습니다.
논문 그림의 D1–D20은 달력 날짜가 표시되지 않으므로 실제 파일 순서와 같다고 단정하지 않습니다.
ANCHOR_DATE로 실제 비교 날짜를 지정하거나 첫 연속 3일에서 시작할 수 있습니다.

기본 NDVI와 함께 **I4 열적 밴드·누적 화재 영역·신규 영역**을 표시합니다.
AF/BA 원시 라벨 그림은 선택 사항입니다. NDVI 합성값이 같더라도 산불이 확산하지 않았다고 해석하지 않습니다.
기록 화재 해석은 아래 NaN 처리 가정에 따릅니다. 누락 파일·격자 불일치는 unknown으로 유지합니다.

기본 설정은 실제 HPC 데이터용입니다. 작성 환경에서 실제 사건은 열 수 없으므로
실제 그림·확산 결과는 HPC 실행 후 확인해야 합니다. DEMO_MODE=True 검증은 별도 합성 예제입니다.


**이 노트북의 주 결과:** 입력 t−1,t와 관측 t+1을 비교하여, 누적 기록 화재 영역과
다음 날 새로 기록된 영역을 표시합니다. 모델 훈련·예측 결과가 아닙니다.
AF/BA는 이 영역을 구성하는 관측 라벨입니다. 순수 BA 증가와 구분합니다.


## 환경

HPC에서 다음 환경을 준비하고 Jupyter의 커널을 **Python (satfire-audit)**로 선택합니다.
Jupyter는 해당 HPC의 지원 방식(Open OnDemand 등)으로 시작하십시오. GPU는 필요 없습니다.

```bash
conda activate satfire-audit
conda install -c conda-forge jupyterlab ipykernel pandas matplotlib nbformat nbclient -y
python -m ipykernel install --user --name satfire-audit --display-name "Python (satfire-audit)"
```

원본 파일은 수정하지 않습니다. 보고서는 저장소의 `audit_results/ndvi_two_day` 아래에 생성됩니다.
같은 설정 재실행 시 이 폴더의 진단 CSV/그림만 갱신합니다. 원본 밖의 경로인지 검사합니다.

In [ ]:
from pathlib import Path
import sys, os, json, datetime as dt
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'scripts'/'satfire_explore.py').is_file()), None)
if REPO is None:
    raise RuntimeError('이 저장소 루트 또는 notebooks 폴더에서 실행하세요.')
sys.path.insert(0, str(REPO/'scripts'))
# git pull updates disk files, but an existing Jupyter kernel retains imported functions.
import importlib, inspect
importlib.invalidate_caches()
_helpers = importlib.reload(importlib.import_module('satfire_explore'))
if 'raw_valid_range' not in inspect.signature(_helpers.read_ndvi).parameters:
    raise RuntimeError('scripts/satfire_explore.py is older than this notebook. Update the repository first.')
print('Helper file:', _helpers.__file__)
print('NDVI reader:', inspect.signature(_helpers.read_ndvi))
from fp_audit import read_fp_split
from satfire_explore import (create_demo_dataset, discover_fp, header_inventory,
    alignment_report, date_index, unique_path, header, grid_reasons, band_stats,
    read_ndvi, progression, progression_series)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

## 1. 설정

제공 NDVI는 전체 27채널 표에서 9번, 별도 FirePred에서는 **1번 밴드라는 전제**를
사용합니다. 실제 밴드 태그와 원본 로더를 아래 표와 대조하십시오.
기본값은 VNP13A1 원본 정수 단위에 맞춰 **NDVI_SCALE=0.0001, NDVI_OFFSET=0.0**입니다.
이는 10,000배 저장 값을 복원하는 처리이며 임의의 0~1 정규화가 아닙니다.
원본 유효 범위 [-10000,10000] 밖의 값(예: fill -15000)은 결측으로 제외합니다.
이미 [-1,1] 실수 단위로 변환한 export라면 NDVI_SCALE=1.0,
PROVIDED_NDVI_RAW_VALID_RANGE=(-1,1)로 바꿔야 하며 중복 스케일링 감지를 유지합니다.

근거: [VNP13 사용자 가이드 8.2/Table 12](https://lpdaac.usgs.gov/documents/1372/VNP13_User_Guide_ATBD_V2.1.2.pdf),
저장소 FP 로더의 NDVI 채널 평균 약 5110.55.
VNP13A1은 **16일 합성, 8일 간격** 제품으로 일별 파일의 NDVI가 같을 수 있습니다.
일별 독립 관측이나 당일 실시간 가용성을 가정하지 말고 합성 기간·배포 시점을 확인해야 합니다.

누적 progression은 CSV의 `legacy_label_sel`을 따릅니다: 0=AF+BA 합집합, 1=AF만.
원본 로더는 raw float에 `logical_or`를 적용합니다. NumPy에서 NaN은 True이므로
NaN 배경을 화재로 누적할 수 있습니다. 여기서는 **유한한 0이 아닌 값만 기록 화재**로
처리하는 보정 방식을 씁니다. `LABEL_NAN_POLICY='no_detection'`은 NaN을
'기록된 탐지 없음'으로 취급한다는 명시적 분석 가정이며, 실제 비소실의 증명이 아닙니다.
BA 전체 NaN도 별도 표기합니다. AF 전체 결측·누락 파일·격자 불일치가 있으면
이후 누적 이력은 불명으로 두며 확산 없음으로 채우지 않습니다.
원본 코드 그대로 누적했을 때의 픽셀 수도 표에 병기해 NaN 영향을 확인합니다.
원본 구현: `legacy/satimg_dataset_processor/satimg_dataset_processor.py`의
`PredDatasetProcessor.pred_dataset_generator_seqtoseq`, `utils.py`의 `read_tiff`.

직접 NDVI를 계산하려면 NDVI_SOURCE='computed'와 REFLECTANCE_CONFIRMED=True가 필요합니다.
VIIRS 입력은 Level-1B 기원이며 복사휘도/스케일을 확인하지 않고 I2/I1 비율을 적용하면 안 됩니다.

In [ ]:
DEMO_MODE = False                         # 합성 데이터: 결과를 실제 산불로 해석하지 않음
DATA_ROOT = Path('/home/jlc3q/data/SatFire/ts-satfire')
ROI_DIR = REPO/'legacy'/'roi'
SPLIT_SCRIPT = REPO/'legacy'/'scripts'/'dataset_gen_pred.py'
EVENT_ID = 'US_2021_NM3323810847220210520'                            # 예: 실제 CSV ID 문자열, 논문 Figure 8/11의 FP 시험 사건
ANCHOR_DATE = None                         # 예: '2021-08-03', 이 날짜가 입력 마지막 날 t
SCAN_ALL_HEADERS = False                    # False이면 선택 사건만 검사, 전체 확인이라고 부르지 않음
NDVI_SOURCE = 'provided'                    # 'provided' 또는 'computed'
PROVIDED_NDVI_BAND = 1                     # 1-based; FirePred 파일 기준
NDVI_SCALE = 0.0001                        # VNP13A1 raw integer -> physical NDVI
NDVI_OFFSET = 0.0
PROVIDED_NDVI_RAW_VALID_RANGE = (-10000, 10000)  # VNP13 native valid range; fill excluded
REFLECTANCE_CONFIRMED = False
BA_BAND = 8                               # 1-based 후보; 실제 확인 필요
BA_RULE = None                            # 이전 BA-only 진단용; 누적 progression에는 사용하지 않음
LABEL_NAN_POLICY = 'no_detection'           # 기록 탐지 정의. 실제 비소실 증명이 아님
SHOW_RAW_LABEL_DIAGNOSTICS = False         # AF/BA 원시값 그림은 기본 숨김
SHOW_STATUS_EXAMPLES = False
PAPER_TIMELINE_FRAMES = 20  # 실제 날짜 순 I4 진단. 논문 D1-D20 대응을 주장하지 않음


In [ ]:
if DEMO_MODE:
    import tempfile
    DEMO_DIR = Path(tempfile.mkdtemp(prefix='satfire_ndvi_demo_'))
    DATA_ROOT, ROI_DIR, SPLIT_SCRIPT = create_demo_dataset(DEMO_DIR)
    EVENT_ID = 'DEMO_FIRE'
    BA_RULE, NDVI_SCALE = 'finite', 1.0
    PROVIDED_NDVI_RAW_VALID_RANGE = (-1, 1)
    print('SYNTHETIC DEMO — NOT ACTUAL WILDFIRE DATA:', DATA_ROOT)
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'HPC 데이터에 접근할 수 없습니다: {DATA_ROOT}')
REPORT_DIR = REPO/'audit_results'/('paper_fp_example_demo' if DEMO_MODE else 'paper_fp_example')
if REPORT_DIR.resolve() == DATA_ROOT.resolve() or DATA_ROOT.resolve() in REPORT_DIR.resolve().parents:
    raise ValueError('보고서는 원본 데이터 밖에 저장해야 합니다.')
REPORT_DIR.mkdir(parents=True, exist_ok=True)
records, split_provenance = read_fp_split(ROI_DIR, SPLIT_SCRIPT)
files = discover_fp(DATA_ROOT, records)
if not files:
    raise RuntimeError('FP 분할 ID와 일치하는 GeoTIFF가 없습니다. 경로와 사건 폴더명을 확인하세요.')
file_table = pd.DataFrame(files)
display(file_table.groupby(['split', 'modality']).size().rename('files').to_frame())
print('Declared FP split:', split_provenance['declared_counts'])

## 2. 사건·이틀 입력 선택

선택은 확산 유무와 관계없이 파일 가용성만으로 합니다. 입력 마지막 날짜 t와 t+1의
NDVI를 비교합니다. t+1의 NDVI는 **사후 진단에만 사용하며 예측 입력에 넣지 않습니다.**

In [ ]:
def available_anchors(index):
    modality = 'FirePred' if NDVI_SOURCE == 'provided' else 'VIIRS_Day'
    return [d for d in sorted(index['VIIRS_Day'])
            if all(unique_path(index, 'VIIRS_Day', d+dt.timedelta(days=k)) for k in [-1, 0, 1])
            and all(unique_path(index, modality, d+dt.timedelta(days=k)) for k in [-1, 0, 1])]

if EVENT_ID is None:
    for candidate in sorted(file_table['event_id'].unique()):
        try:
            candidate_index = date_index(files, candidate)
        except ValueError:
            continue
        if available_anchors(candidate_index):
            EVENT_ID = candidate
            break
if not DEMO_MODE and EVENT_ID == 'US_2021_NM3323810847220210520' and records[EVENT_ID]['split'] != 'test':
    raise ValueError('논문 예시가 FP 시험 분할인지 확인하세요.')
if EVENT_ID is None or EVENT_ID not in records:
    raise ValueError('이틀 이력과 다음 날 관측이 있는 사건을 지정하세요.')
index = date_index(files, EVENT_ID)
anchors = available_anchors(index)
if not anchors:
    raise ValueError('선택 사건에서 t−1, t, t+1 연속 관측을 구성할 수 없습니다.')
t = dt.date.fromisoformat(ANCHOR_DATE) if ANCHOR_DATE else anchors[0]
if t not in anchors:
    raise ValueError(f'지정 날짜는 이틀 입력·다음 날 비교가 불가능합니다. 예시: {anchors[:8]}')
dates = [t-dt.timedelta(days=1), t, t+dt.timedelta(days=1)]
display(pd.DataFrame([dict(**records[EVENT_ID], selected_t=str(t))]))
print('Input:', dates[:2], ' | Retrospective next-day comparison:', dates[2])
anchor_table = pd.DataFrame([dict(
    input_start=str(d-dt.timedelta(days=1)), input_end=str(d),
    next_day=str(d+dt.timedelta(days=1)),
    days_from_roi_start=(d-dt.date.fromisoformat(records[EVENT_ID]['roi_start'])).days
) for d in anchors])
display(anchor_table.head(25))
print('legacy_label_sel:', records[EVENT_ID]['legacy_label_sel'])
print('label_sel=0: reference loader uses cumulative AF/BA union; label_sel=1: cumulative AF.')
print('Cumulative recorded-fire comparison follows label_sel, correcting NaN truthiness; see methods.')


## 3. 영상 크기가 실제로 같은지 확인

원본 GeoTIFF 크기와 모델에서 자르는 256×256 크기는 구분합니다.
아래는 선택 범위의 **모든 파일 헤더**를 읽습니다. 첫 실행은 파일 수에 따라 시간이 걸릴 수 있습니다.
동일 shape라도 CRS가 없거나 transform이 다르면 같은 위치라는 확인은 끝나지 않은 것입니다.
자동 재투영·강제 CRS 지정·원본 수정은 하지 않습니다.

In [ ]:
scope_files = files if SCAN_ALL_HEADERS else [r for r in files if r['event_id'] == EVENT_ID]
headers = header_inventory(scope_files)
header_table = pd.DataFrame(headers)
readable = header_table[header_table['error'].isna()].copy()
readable['height'] = readable['shape'].map(lambda x: x[0])
readable['width'] = readable['shape'].map(lambda x: x[1])
readable['crs_display'] = readable['crs'].fillna('MISSING')
display(readable.groupby(['modality', 'height', 'width', 'crs_display']).size().rename('files').to_frame())
n_shapes = readable[['height', 'width']].drop_duplicates().shape[0]
print('Scope:', 'ALL SELECTED FP FILES' if SCAN_ALL_HEADERS else 'SELECTED EVENT ONLY')
print('Requested/readable/unreadable:', len(scope_files), len(readable), len(headers)-len(readable))
print('Distinct spatial shapes:', n_shapes)
if len(readable) == len(scope_files) and n_shapes == 1:
    print('검사 범위의 모든 파일은 동일한 height/width입니다. 위치 정합은 아래에서 별도 확인합니다.')
else:
    print('크기가 여러 종류이거나 읽지 못한 파일이 있습니다. 동일 크기라고 확정하지 않습니다.')
alignment = pd.DataFrame(alignment_report(headers))
display(alignment.groupby(['modality', 'reasons']).size().rename('same_day_pairs').to_frame())
display(alignment[alignment['reasons'] != 'aligned'].head(12))
header_table.to_csv(REPORT_DIR/'all_fp_headers.csv', index=False)
alignment.to_csv(REPORT_DIR/'same_day_grid_alignment.csv', index=False)

## 4. 밴드·스케일·라벨 원시값 확인

현재 고른 3일의 값을 확인한 뒤 위 설정을 수정할 수 있습니다.
밴드명 부재나 스케일=1은 값의 의미가 검증됐다는 뜻이 아닙니다.
원시값 통계는 NaN을 보존합니다. 누적 기록 지도에서의 NaN 가정은 위 방법 절을 따릅니다.

In [ ]:
raw_rows = []
for d in dates:
    day_path = unique_path(index, 'VIIRS_Day', d)
    ndvi_path = unique_path(index, 'FirePred', d)
    for kind, path, band in [('BA candidate', day_path, BA_BAND),
                             ('provided NDVI candidate', ndvi_path, PROVIDED_NDVI_BAND)]:
        if path:
            try:
                raw_rows.append(dict(date=str(d), kind=kind, **band_stats(path, band)))
            except Exception as exc:
                raw_rows.append(dict(date=str(d), kind=kind, error=str(exc)))
raw_stats = pd.DataFrame(raw_rows)
display(raw_stats)
raw_stats.to_csv(REPORT_DIR/'selected_raw_band_stats.csv', index=False)

## 누적 라벨 진단

`cumulative_pixels`는 NaN을 보정한 누적 기록 픽셀, `literal_legacy_pixels`는
원본 NumPy 논리식을 그대로 쓴 진단값입니다. 서로 다르면 NaN 처리 영향을 확인하세요.
`ba_all_nan=True`는 BA에 기록이 없음을 보여주며 실제 무소실을 증명하지 않습니다.


In [ ]:
import importlib
progression_helpers=importlib.reload(importlib.import_module('fp_progression'))
compare_days=progression_helpers.compare_days
label_sel=int(records[EVENT_ID]['legacy_label_sel'])
if LABEL_NAN_POLICY != 'no_detection':
    raise ValueError('Only the explicitly documented no_detection policy is implemented.')
cumulative_rows,cumulative_maps=progression_helpers.build_progression(
    index,label_sel,dt.date.fromisoformat(records[EVENT_ID]['roi_start']),LABEL_NAN_POLICY)
cumulative_table=pd.DataFrame(cumulative_rows)
display(cumulative_table.head(40))
cumulative_table.to_csv(REPORT_DIR/'cumulative_label_diagnostics.csv',index=False)
if (pd.to_numeric(cumulative_table['literal_legacy_pixels'],errors='coerce') !=
        pd.to_numeric(cumulative_table['cumulative_pixels'],errors='coerce')).any():
    print('WARNING: literal legacy NaN truthiness differs from finite-nonzero recorded masks. See methods.')
print('AF/BA build progression labels here; these are observations, not model predictions.')


## 5. NDVI 비교와 변화량

색 범위는 날짜마다 고정합니다. 검증 가능한 동일 격자에서 공통 유효 픽셀만 차분합니다.
NDVI 감소는 구름·연무·시야각·소실 등에 의해서도 발생하므로 신규 소실 정답으로 사용하지 않습니다.
제공 NDVI 제품은 합성 관측일 수 있어 파일 날짜가 하루 달라도 값이 동일할 수 있습니다.

In [ ]:
def plot_pair(current, save_name=None):
    # Refresh disk implementation and avoid stale `from module import function` aliases.
    import importlib, inspect
    importlib.invalidate_caches()
    helpers = importlib.reload(importlib.import_module('satfire_explore'))
    expected = (REPO/'scripts'/'satfire_explore.py').resolve()
    if Path(helpers.__file__).resolve() != expected:
        raise RuntimeError(f'Unexpected helper module: {helpers.__file__}; expected {expected}')
    ndvi_reader = helpers.read_ndvi
    if 'raw_valid_range' not in inspect.signature(ndvi_reader).parameters:
        raise RuntimeError('On-disk NDVI helper is outdated. Update scripts/satfire_explore.py from main.')
    sample_dates = [current-dt.timedelta(days=1), current, current+dt.timedelta(days=1)]
    modality = 'FirePred' if NDVI_SOURCE == 'provided' else 'VIIRS_Day'
    paths = [helpers.unique_path(index, modality, d) for d in sample_dates]
    observations = [ndvi_reader(p, source=NDVI_SOURCE, band=PROVIDED_NDVI_BAND,
        scale=NDVI_SCALE, offset=NDVI_OFFSET, reflectance_confirmed=REFLECTANCE_CONFIRMED,
        raw_valid_range=PROVIDED_NDVI_RAW_VALID_RANGE) for p in paths]
    display(pd.DataFrame([dict(date=str(d), **{k:v for k,v in obs.items() if k != 'array'})
                          for d, obs in zip(sample_dates, observations)]))
    fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
    images = []
    for ax, date, obs in zip(axes[:3], sample_dates, observations):
        arr = obs['array']
        if arr is None:
            ax.text(.5, .5, obs['status'], ha='center', va='center', wrap=True, transform=ax.transAxes)
        else:
            images.append(ax.imshow(arr, vmin=-1, vmax=1, cmap='viridis'))
        ax.set_title(f'{date} | '+('input t-1' if date < current else 'input t' if date == current else 'observed t+1'))
        ax.set_xlabel('Column (native pixels)')
        ax.set_ylabel('Row (native pixels)')
    if images:
        fig.colorbar(images[0], ax=list(axes[:3]), shrink=.8, label='NDVI')
    before_path = helpers.unique_path(index, 'VIIRS_Day', current)
    after_path = helpers.unique_path(index, 'VIIRS_Day', sample_dates[2])
    change = helpers.progression(before_path, after_path, BA_BAND, BA_RULE)
    delta, common = None, None
    if observations[1]['array'] is not None and observations[2]['array'] is not None:
        reasons = helpers.grid_reasons(helpers.header(paths[1]), helpers.header(paths[2]))
        if not reasons:
            a, b = observations[1]['array'], observations[2]['array']
            common = np.isfinite(a) & np.isfinite(b)
            delta = np.where(common, b-a, np.nan)
            image = axes[3].imshow(delta, vmin=-2, vmax=2, cmap='RdBu_r')
            fig.colorbar(image, ax=axes[3], shrink=.8, label='NDVI(t+1) - NDVI(t)')
            overlay_ok = (change['new_mask'] is not None and before_path is not None
                and not helpers.grid_reasons(helpers.header(before_path), helpers.header(paths[1])))
            if overlay_ok and change['new_mask'].any() and not change['new_mask'].all():
                axes[3].contour(change['new_mask'].astype(float), levels=[.5], colors=['black'], linewidths=.8)
            print('Common valid NDVI pixels:', int(common.sum()), '/', common.size)
        else:
            axes[3].text(.5, .5, 'No pixel subtraction: '+', '.join(reasons), ha='center', va='center', wrap=True, transform=axes[3].transAxes)
    else:
        axes[3].text(.5, .5, 'NDVI unavailable: inspect raw values and scale', ha='center', va='center', wrap=True, transform=axes[3].transAxes)
    axes[3].set_title('Change; black contour = new recorded fire')
    axes[3].set_xlabel('Column (native pixels)')
    axes[3].set_ylabel('Row (native pixels)')
    fig.suptitle(f'{EVENT_ID} | '+('SYNTHETIC | ' if DEMO_MODE else '')+change['status']+' | observed labels, not model predictions')
    if save_name:
        fig.savefig(REPORT_DIR/f'{save_name}.png', dpi=150)
    plt.show()
    print({k:v for k,v in change.items() if not isinstance(v, np.ndarray)})
    return observations, change

observations, selected_change = plot_pair(t, 'selected_ndvi_pair')

## 6. 누적 화재 영역과 다음 날 추가 영역 — 주 결과

왼쪽부터 입력 마지막 날 I4, **t까지 누적 기록 화재**, **t+1까지 누적 기록 화재**,
**t+1 신규 기록 영역**입니다. 신규 영역은 `C(t+1) & ~C(t)`입니다.
누적은 ROI 시작일부터 하며 t+1 라벨은 정답·사후 비교에만 씁니다.
두 날 입력 길이와 누적 라벨 이력 길이는 다릅니다.

원본 로더는 `[128:384,128:384]` crop과 슬라이딩 윈도우 누적 상태를 사용합니다.
이 노트북은 원본 전체 격자의 날짜순 누적 비교이므로 정확한 벤치마크 재현은 아닙니다.
'신규 0'은 이 자료의 라벨 정의에 따른 결과이며 실제 확산이 없었다는 보장이 아닙니다.


In [ ]:
import rasterio

def read_raw_band(path, band):
    if path is None:
        return None, 'missing/duplicate file'
    try:
        with rasterio.open(path) as ds:
            if not 1 <= band <= ds.count:
                return None, f'band {band} absent (count={ds.count})'
            arr = ds.read(band, masked=True).astype('float64').filled(np.nan)
            arr[~np.isfinite(arr)] = np.nan
            return arr, 'available' if np.isfinite(arr).any() else 'all missing'
    except Exception as exc:
        return None, str(exc)

from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
change = compare_days(cumulative_maps, t)
thermal, thermal_status = read_raw_band(unique_path(index,'VIIRS_Day',t),4)
fig, axes = plt.subplots(1,4,figsize=(17,4),constrained_layout=True)
if thermal is not None and np.isfinite(thermal).any():
    lo,hi=np.nanpercentile(thermal,[1,99])
    axes[0].imshow(thermal,cmap='gray',vmin=lo,vmax=hi)
else:
    axes[0].text(.5,.5,thermal_status,ha='center',transform=axes[0].transAxes)
axes[0].set_title(f'{t} | observed I4')
for ax,key,title,color in zip(axes[1:], ['current_mask','next_mask','new_mask'],
        [f'Cumulative through {t}',f'Cumulative through {t+dt.timedelta(days=1)}',
         f'New on {t+dt.timedelta(days=1)}'], ['#315b85','#315b85','#c44e00']):
    mask=change[key]
    if mask is None:
        ax.text(.5,.5,change['status'],ha='center',va='center',wrap=True,transform=ax.transAxes)
    else:
        ax.imshow(mask,cmap=ListedColormap(['#f3f3f3',color]),vmin=0,vmax=1,interpolation='nearest')
        ax.legend(handles=[Patch(facecolor=color,label=f'Recorded fire: {int(mask.sum()):,} pixels')],loc='lower left',fontsize=8)
    ax.set_title(title)
for ax in axes:
    ax.set_xlabel('Column (native pixels)')
    ax.set_ylabel('Row (native pixels)')
fig.suptitle(f'{EVENT_ID} | '+('SYNTHETIC | ' if DEMO_MODE else '')+change['status']+' | observed labels')
fig.savefig(REPORT_DIR/'cumulative_progression_pair.png',dpi=150)
plt.show()
print('Next-day new recorded-fire pixels:',change['new_pixels'])

# Optional raw AF/BA panels for troubleshooting; not a separate task or model output.
if SHOW_RAW_LABEL_DIAGNOSTICS:
    current_path = unique_path(index, 'VIIRS_Day', t)
    next_path = unique_path(index, 'VIIRS_Day', t+dt.timedelta(days=1))
    raw_specs = [
        (current_path, 4, f'{t} | I4 stored values'),
        (next_path, 4, f'{t+dt.timedelta(days=1)} | I4 stored values'),
        (next_path, 7, f'{t+dt.timedelta(days=1)} | AF raw band 7'),
        (next_path, BA_BAND, f'{t+dt.timedelta(days=1)} | BA raw band {BA_BAND}'),
    ]
    raw_results = [read_raw_band(p,b) for p,b,_ in raw_specs]
    thermal_values = [a[np.isfinite(a)] for a,status in raw_results[:2]
                      if a is not None and np.isfinite(a).any()]
    thermal_limits = np.percentile(np.concatenate(thermal_values), [1,99]) if thermal_values else None
    fig, axes = plt.subplots(1,4,figsize=(16,4),constrained_layout=True)
    for i,(ax,(_,_,title),(arr,status)) in enumerate(zip(axes,raw_specs,raw_results)):
        if arr is None or not np.isfinite(arr).any():
            ax.text(.5,.5,status,ha='center',va='center',transform=ax.transAxes)
        else:
            options = dict(cmap='gray' if i < 2 else 'viridis')
            if i < 2 and thermal_limits is not None:
                options.update(vmin=thermal_limits[0], vmax=thermal_limits[1])
            im=ax.imshow(arr, **options)
            fig.colorbar(im, ax=ax, shrink=.8, label='Stored value')
        ax.set_title(title)
        ax.set_xlabel('Column (native pixels)')
        ax.set_ylabel('Row (native pixels)')
    fig.suptitle(f'{EVENT_ID} | '+('SYNTHETIC | ' if DEMO_MODE else '')+'Raw thermal/label diagnostics')
    fig.savefig(REPORT_DIR/'paper_example_raw_bands.png',dpi=150)
    plt.show()


## 실제 날짜 순 I4 시퀀스

아래는 로컬 관측 파일의 날짜 순서입니다. 논문의 D1–D20과 대응시켰다고 주장하지 않습니다.
원 논문은 누적 열적 정보 및 모델 예측을 표시하지만, 이 진단은 **날짜별 I4 원시값**을 표시합니다.
색 범위는 선택한 프레임 전체의 1–99 백분위로 고정하며, 표시용 대비 조정은 원본 값을 바꾸지 않습니다.
첫 20프레임을 기본으로 하며 `PAPER_TIMELINE_FRAMES`를 바꿀 수 있습니다.


In [ ]:
timeline_dates = sorted(index['VIIRS_Day'])[:PAPER_TIMELINE_FRAMES]
frames = [read_raw_band(unique_path(index,'VIIRS_Day',d),4) for d in timeline_dates]
finite_values = [a[np.isfinite(a)][::max(1,a.size//10000)] for a,s in frames
                 if a is not None and np.isfinite(a).any()]
limits = np.percentile(np.concatenate(finite_values),[1,99]) if finite_values else None
ncols=5
nrows=max(1,(len(frames)+ncols-1)//ncols)
fig,axes=plt.subplots(nrows,ncols,figsize=(15,3*nrows),squeeze=False,constrained_layout=True)
last_image=None
for ax in axes.flat:
    ax.set_axis_off()
for ax,d,(arr,status) in zip(axes.flat,timeline_dates,frames):
    ax.set_axis_on()
    if arr is None or not np.isfinite(arr).any():
        ax.text(.5,.5,status,ha='center',va='center',transform=ax.transAxes)
    else:
        last_image=ax.imshow(arr,cmap='gray',vmin=limits[0],vmax=limits[1])
    ax.set_title(str(d))
    ax.set_xticks([])
    ax.set_yticks([])
if last_image is not None:
    fig.colorbar(last_image,ax=list(axes.flat),shrink=.5,label='I4 stored value (common display scale)')
fig.suptitle(f'{EVENT_ID} | '+('SYNTHETIC | ' if DEMO_MODE else '')+'Local calendar sequence, not paper D-label mapping')
fig.savefig(REPORT_DIR/'paper_example_i4_sequence.png',dpi=150)
plt.show()


## 7. 신규 영역이 없는 날짜도 유지

모든 날짜쌍을 남깁니다. `no_new_recorded_fire_pixels`는 유효한 기록 라벨에서 신규 0,
`unknown_*`는 누락·정합·불완전 이력으로 비교할 수 없는 경우입니다.
불명은 그래프의 빈 구간이며 0으로 대체하지 않습니다.
BA NaN 및 원본 코드의 NaN→True 누적 영향은 앞의 진단표를 확인하십시오.


In [ ]:
series_rows=[]
for current in sorted(index['VIIRS_Day']):
    result=compare_days(cumulative_maps,current)
    series_rows.append(dict(current_date=str(current),target_date=str(current+dt.timedelta(days=1)),
        two_day_history=current in anchors,status=result['status'],new_pixels=result['new_pixels']))
series=pd.DataFrame(series_rows)
display(series.head(40))
display(series.groupby(['two_day_history','status']).size().rename('date_pairs').to_frame())
series.to_csv(REPORT_DIR/'progression_by_day.csv',index=False)
fig,ax=plt.subplots(figsize=(11,3.5),constrained_layout=True)
values=pd.to_numeric(series['new_pixels'],errors='coerce')
ax.plot(pd.to_datetime(series['target_date']),values,marker='o',label='New recorded-fire pixels (gaps = unknown)')
ax.set(xlabel='Target date t+1',ylabel='New recorded-fire pixels',title='Observed cumulative-label increments; not model predictions')
ax.grid(alpha=.2)
ax.legend()
fig.savefig(REPORT_DIR/'recorded_progression_by_day.png',dpi=150)
plt.show()


## 판단과 다음 단계

- AF/BA는 progression 정답을 구성하는 관측 정보입니다. 그림은 예측 결과가 아닙니다.
- NDVI(t+1)와 t+1 화재 라벨은 사후 비교·평가 전용입니다. 모델 입력에는 t까지의 정보만 넣습니다.
- 신규 기록 0인 날도 유지하되, 실제 무확산과 관측 누락을 동일시하지 않습니다.
- NaN 배경 보정 및 원본 전체 격자 누적 때문에 논문 학습 표본을 그대로 재현한 결과는 아닙니다.
- AF+BA 신규 기록은 순수한 '내일 신규 소실'과 다를 수 있습니다. 연구의 목표 라벨을 별도로 확정해야 합니다.


In [ ]:
provenance = dict(demo=DEMO_MODE, data_root=str(DATA_ROOT), event_id=EVENT_ID,
    anchor_date=str(t), input_days=2, ndvi_source=NDVI_SOURCE,
    provided_ndvi_band=PROVIDED_NDVI_BAND, ndvi_scale=NDVI_SCALE,
    ndvi_offset=NDVI_OFFSET, ndvi_raw_valid_range=PROVIDED_NDVI_RAW_VALID_RANGE,
    reflectance_confirmed=REFLECTANCE_CONFIRMED,
    ba_band=BA_BAND, ba_rule=BA_RULE, label_nan_policy=LABEL_NAN_POLICY,
    progression_method="finite_nonzero_cumulative_recorded_fire", label_sel=label_sel,
    accumulation_start=records[EVENT_ID]["roi_start"], spatial_scope="native_full_grid", all_fp_headers_requested=SCAN_ALL_HEADERS,
    header_requested_count=len(scope_files), header_readable_count=len(readable),
    split_provenance=split_provenance)
(REPORT_DIR/'notebook_provenance.json').write_text(json.dumps(provenance, indent=2, ensure_ascii=False), encoding='utf-8')
print('Reports:', REPORT_DIR)